In [1]:
from pyspark.sql import SparkSession  
import getpass 
username = getpass 
spark = SparkSession.builder.config('spark.ui.port','0').\
        config('spark.sql.warehouse.dir',f"/user/itv022063/warehouse").\
        enableHiveSupport().\
        master('yarn').\
        getOrCreate()

In [2]:
spark.sql("use itv022063_lending_club")

""


In [3]:
bad_data_customers=spark.sql("""
select member_id from 
(
select member_id , 
count(*) as total
from itv022063_lending_club.customers 
group by member_id 
having total >1
)
""")

In [4]:
bad_data_loans_defaulters_detail_rec_enq=spark.sql("""
select member_id from 
(
select member_id , 
count(*) as total
from itv022063_lending_club.loans_defaulters_detail_rec_enq 
group by member_id 
having total >1
)
""")

In [5]:
bad_data_loans_defaulters_delinq=spark.sql("""
select member_id from 
(
select member_id , 
count(*) as total
from itv022063_lending_club.loans_defaulters_delinq 
group by member_id 
having total >1
)
""")

In [6]:
bad_data_customers.repartition(1).write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/bad_data/bad_data_customers_parquet") \
.save()

In [12]:
bad_data_customers.repartition(1).write \
.format("csv") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/bad_data/bad_data_customers_csv") \
.save()

In [15]:
bad_data_loans_defaulters_detail_rec_enq.repartition(1).write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/bad_data/bad_data_loans_defaulters_detail_rec_enq_parquet") \
.save()

In [18]:
bad_data_loans_defaulters_detail_rec_enq.repartition(1).write \
.format("csv") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/bad_data/bad_data_loans_defaulters_detail_rec_enq_csv") \
.save()

In [21]:
bad_data_loans_defaulters_delinq.repartition(1).write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/bad_data/bad_data_loans_defaulters_delinq_parquet") \
.save()

In [23]:
bad_data_loans_defaulters_delinq.repartition(1).write \
.format("csv") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/bad_data/bad_data_loans_defaulters_delinq_csv") \
.save()

In [24]:
bad_data_customers.printSchema()

root
 |-- member_id: string (nullable = true)



In [31]:
bad_customer_data_df = bad_data_customers.select("member_id") \
.union(bad_data_loans_defaulters_detail_rec_enq.select("member_id")) \
.union(bad_data_loans_defaulters_delinq.select("member_id"))

In [32]:
bad_customer_data_final_df = bad_customer_data_df.distinct()

In [33]:
bad_customer_data_final_df.count()

3189

In [34]:
bad_customer_data_final_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/bad_data/bad_customer_data_final") \
.save()

In [35]:
bad_customer_data_final_df.createOrReplaceTempView("bad_data_customer")

In [36]:
customers_df = spark.sql("""select * from itv022063_lending_club.customers
where member_id NOT IN (select member_id from bad_data_customer)
""")

In [37]:
customers_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/cleaned_new/customers_parquet") \
.save()

In [38]:
loans_defaulters_delinq_df = spark.sql("""select * from itv006277_lending_club.loans_defaulters_delinq
where member_id NOT IN (select member_id from bad_data_customer)
""")

In [39]:
loans_defaulters_delinq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/cleaned_new/loans_defaulters_delinq_parquet") \
.save()

In [41]:
loans_defaulters_detail_rec_enq_df = spark.sql("""select * from itv022063_lending_club.loans_defaulters_detail_rec_enq
where member_id NOT IN (select member_id from bad_data_customer)
""")

In [42]:
loans_defaulters_detail_rec_enq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/cleaned_new/loans_defaulters_detail_rec_enq_parquet") \
.save()

In [43]:
spark.sql("""
create EXTERNAL TABLE itv022063_lending_club.customers_new(member_id string, emp_title string, emp_length int, home_ownership string, 
annual_income float, address_state string, address_zipcode string, address_country string, grade string, 
sub_grade string, verification_status string, total_high_credit_limit float, application_type string, 
join_annual_income float, verification_status_joint string, ingest_date timestamp)
stored as parquet
LOCATION '/user/itv022063/Yash/cleaned_new/customers_parquet'
""")

""


In [44]:
spark.sql("""
create EXTERNAL TABLE itv022063_lending_club.loans_defaulters_delinq_new(member_id string,delinq_2yrs integer, delinq_amnt float, mths_since_last_delinq integer)
stored as parquet
LOCATION '/user/itv022063/Yash/cleaned_new/loans_defaulters_delinq_parquet'
""")

""


In [45]:
spark.sql("""
create EXTERNAL TABLE itv022063_lending_club.loans_defaulters_detail_rec_enq_new(member_id string, pub_rec integer, pub_rec_bankruptcies integer, inq_last_6mths integer)
stored as parquet
LOCATION '/user/itv022063/Yash/cleaned_new/loans_defaulters_detail_rec_enq_parquet'
""")

""


In [47]:
spark.sql("""select member_id, count(*) as total 
from itv022063_lending_club.customers_new
group by member_id order by total desc""")

member_id,total
3df3cdeddb74a8712...,1
0b6beb388edd047d8...,1
c02f5f84058c5f952...,1
0b9083d10c1a1db0b...,1
57cf658ad87be221b...,1
576209768213e3f3b...,1
dfcbd6434a458f952...,1
c2da27a4d1f99fba9...,1
38c9e9077eac24f54...,1
a85ccb7523cd134c1...,1


In [48]:
spark.stop()